# S3.9 — cast, alias, lit, col
**Date completed:** October 2026  
**Status:** Completed  
**Key concept:** Building blocks of every column expression

In [0]:
# ============================================================
# S3.9 — Cell 2
# col(), alias(), cast(), lit() in Action
#
# Goal:
# 1. Use col() to reference DataFrame columns.
# 2. Use alias() to name columns and expressions.
# 3. Use cast() to standardize data types.
# 4. Understand safe handling of invalid type conversions.
# 5. Use lit() to create constant-value columns.
# 6. Combine col(), alias(), cast(), and lit()
#    in a Bronze → Silver transformation pattern.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql.types import (
    LongType,
    DecimalType
)

print("=== S3.9: col(), alias(), cast(), lit() ===")
print()


# ------------------------------------------------------------
# Step 1 — Load source table
# ------------------------------------------------------------

df_orders = spark.table(
    "samples.tpch.orders"
)

print("Orders DataFrame loaded.")
print()


# ------------------------------------------------------------
# Step 2 — col(): Reference DataFrame Columns
#
# There are several ways to reference/select columns.
#
# F.col("column_name")
#     → Returns a Spark Column expression.
#
# df["column_name"]
#     → Also returns a Spark Column expression.
#
# "column_name"
#     → A plain column-name string accepted by many
#       DataFrame APIs such as select(), drop(), groupBy(),
#       orderBy(), etc.
#
# F.col() is especially useful when building expressions
# involving functions, calculations, cast(), alias(), etc.
# ------------------------------------------------------------

print("=== STEP 2: col() ===")


# Method 1 — F.col()

df_col1 = df_orders.select(
    F.col("o_orderkey"),
    F.col("o_totalprice")
)


# Method 2 — DataFrame column reference

df_col2 = df_orders.select(
    df_orders["o_orderkey"],
    df_orders["o_totalprice"]
)


# Method 3 — Plain string column names

df_col3 = df_orders.select(
    "o_orderkey",
    "o_totalprice"
)


print(f"F.col() columns : {df_col1.columns}")

print(f"df['column'] columns : {df_col2.columns}")

print(f"String-name columns    : {df_col3.columns}")

assert df_col1.columns == df_col2.columns
assert df_col1.columns == df_col3.columns

print("Validation passed: all three approaches " "selected the same columns.")

print()













In [0]:
# ------------------------------------------------------------
# Important col() Concept
#
# F.col("o_totalprice") does NOT fetch the data immediately.
#
# It creates a Spark Column expression that can participate
# in transformations such as:
#
# arithmetic
# cast()
# alias()
# when()
# round()
# filter()
# ------------------------------------------------------------

# ------------------------------------------------------------
# Step 3 — alias(): Name Columns and Expressions
#
# alias() assigns a name to the resulting Column expression.
#
# It can rename:
#
# 1. An existing column
# 2. A calculated expression
# ------------------------------------------------------------

print("=== STEP 3: alias() ===")

df_aliased = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_custkey").alias("customer_id"),

    F.col("o_orderstatus").alias("status"),

    F.col("o_totalprice").alias("total_price"),

    F.col("o_orderdate").alias("order_date")
)

print("\nRenamed output columns:")

df_aliased.show(3, truncate = False)

# ------------------------------------------------------------
# alias() can also name a calculated expression.
# ------------------------------------------------------------

df_expression_alias = df_orders.select(

    F.col("o_orderkey").alias("order_id"),
    (F.col("o_totalprice") * 1.18).alias("price_with_tax")
)


print("\nalias() applied to a calculated expression:")

df_expression_alias.show(3,truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 4 — cast(): Standardize Data Types
#
# Bronze data often arrives as strings even though the
# business values represent:
#
# IDs
# dates
# decimals
# booleans
#
# cast() converts a Column expression to another Spark type.
#
# NOTE:
# cast() does not guarantee that every source value is valid
# for the requested target type.
# ------------------------------------------------------------

print("=== STEP 4: cast() ===")
# Simulated Bronze data where source fields arrive as STRING.

bronze_data = [

    (
        "1001",
        "201",
        "2500.50",
        "2026-01-15",
        "true"
    ),

    (
        "1002",
        "202",
        "1800.00",
        "2026-01-16",
        "false"
    ),

    (
        "1003",
        "203",
        "3200.75",
        "2026-01-17",
        "true"
    )
]

df_bronze = spark.createDataFrame(
    bronze_data,
    [
        "order_id",
        "customer_id",
        "total_price",
        "order_date",
        "is_active"
    ]
)

print("\nBronze schema:")

df_bronze.printSchema()

print("\nBronze records:")

df_bronze.show(truncate=False)
# ------------------------------------------------------------
# Convert Bronze STRING fields to target Silver data types.
#
# IDs:
# BIGINT / LongType
#
# Financial amount:
# DecimalType(18,2)
#
# Date:
# date
#
# Flag:
# boolean
# ------------------------------------------------------------

df_silver = (
    df_bronze

    .withColumn("order_id", F.col("order_id").cast(LongType()))
    .withColumn("customer_id", F.col("customer_id").cast(LongType()))
    .withColumn("total_price", F.col("total_price").cast(DecimalType(18, 2)))
    .withColumn("order_date", F.col("order_date").cast("date"))
    .withColumn("is_active", F.col("is_active").cast("boolean")))

print("\nSilver schema after cast():")

df_silver.printSchema()

print("\nSilver records:")

df_silver.show(truncate=False)
print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Invalid Type Conversion
#
# cast() assumes the input can be converted to the target type.
#
# Invalid values can cause problems depending on Spark /
# ANSI behaviour.
#
# When malformed input should safely become NULL rather than
# fail the operation, Databricks/Spark SQL provides try_cast().
#
# We use a separate learning DataFrame so the main Silver
# transformation is not affected.
# ------------------------------------------------------------

print("=== STEP 5: INVALID CAST HANDLING ===")

invalid_cast_data = [

    ("1001", "2500.50"),

    ("BAD_ID", "1800.00"),

    ("1003", "NOT_A_PRICE")
]

df_invalid_cast = spark.createDataFrame(
    invalid_cast_data,
    [
        "order_id",
        "total_price"
    ]
)


print("\nSource values:")

df_invalid_cast.show(
    truncate=False
)

# ------------------------------------------------------------
# try_cast():
#
# Valid conversion
#     → converted value
#
# Invalid conversion
#     → NULL
#
# This allows us to identify conversion failures and
# quarantine or validate them separately.
# ------------------------------------------------------------

df_try_cast = df_invalid_cast.select(

    "order_id",
    "total_price",
    F.expr("try_cast(order_id AS BIGINT)").alias("order_id_cast"
    ),

    F.expr("try_cast(total_price AS DECIMAL(18,2))").alias("total_price_cast")
)

print("\nSafe conversion using try_cast():")

df_try_cast.show(truncate=False)


print("""
Important:

cast()
→ Converts to the requested data type.
→ Invalid input may fail depending on conversion and ANSI behaviour.

try_cast()
→ Attempts the conversion.
→ Returns NULL when a supported conversion fails because
  the value is malformed.

After try_cast(), business/data-quality validation is still
required to determine whether the NULL is acceptable.
""")

In [0]:
# ------------------------------------------------------------
# Step 6 — lit(): Add Constant Columns
#
# lit() creates a Spark Column containing a literal value.
#
# Every output row receives that value.
#
# Common enterprise uses:
#
# source tracking
# batch metadata
# pipeline metadata
# flags
# version information
# ------------------------------------------------------------

print("=== STEP 6: lit() ===")

df_with_metadata = (
    df_silver
    .withColumn("source_system", F.lit("TPCH"))
    .withColumn("batch_date", F.lit("2026-10-12").cast("date"))
    .withColumn("batch_id", F.lit(20261012))
    .withColumn("is_processed",F.lit(False))
    .withColumn("pipeline_version", F.lit("1.0.0"))
)

print("\nDataFrame with metadata columns:")

df_with_metadata.printSchema()

df_with_metadata.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Combine col(), cast(), alias(), and lit()
#
# This demonstrates a common Bronze → Silver style:
#
# col()
#     → reference source field
#
# cast()
#     → standardize target data type
#
# alias()
#     → standardize target column name
#
# lit()
#     → add constant metadata
# ------------------------------------------------------------

print("=== STEP 7: COMBINED SILVER TRANSFORMATION ===")

df_silver_final = df_orders.select(

    F.col("o_orderkey").cast("long").alias("order_id"),

    F.col("o_custkey").cast("long").alias("customer_id"),

    F.col("o_orderstatus").alias("status"),

    F.col("o_totalprice").cast("decimal(18,2)").alias("total_price"),

    F.col("o_orderdate").cast("date").alias("order_date"),

    F.lit("TPCH").alias("source_system"),

    F.lit(False).alias("is_deleted")
)

print("\nSilver-style schema:")

df_silver_final.printSchema()

print("\nSample Silver-style records:")

df_silver_final.show(5, truncate=False)

silver_count = df_silver_final.count()

print(f"\nTotal rows: {silver_count:,}")
print()

In [0]:
# ------------------------------------------------------------
# Step 8 — Validate Resulting Schema
# ------------------------------------------------------------

print("=== STEP 8: VALIDATION ===")


expected_columns = [

    "order_id",
    "customer_id",
    "status",
    "total_price",
    "order_date",
    "source_system",
    "is_deleted"
]

assert df_silver_final.columns == expected_columns, (
    "Silver column structure does not match expectation"
)

schema_map = {
    field.name: field.dataType.simpleString()
    for field in df_silver_final.schema.fields
}


assert schema_map["order_id"] == "bigint", ("order_id is not BIGINT")

assert schema_map["customer_id"] == "bigint", ("customer_id is not BIGINT")

assert schema_map["total_price"] == "decimal(18,2)", ("total_price is not DECIMAL(18,2)")

assert schema_map["order_date"] == "date", ("order_date is not DATE")

assert schema_map["source_system"] == "string", ("source_system is not STRING")

assert schema_map["is_deleted"] == "boolean", ("is_deleted is not BOOLEAN")

print("Schema validation passed.")
print()


In [0]:
# ------------------------------------------------------------
# Step 9 — Transformations vs Actions
# ------------------------------------------------------------

print("=== STEP 9: TRANSFORMATIONS VS ACTIONS ===")

print("""
Column-expression building blocks:

col()
alias()
cast()
lit()

They are normally used inside DataFrame transformations such as:

select()
withColumn()

They do not independently trigger execution.

Transformations used:

select()
withColumn()

Actions used:

show()
count()
printSchema() displays schema metadata but does not perform
a full row computation like count().

Important:

Spark builds expressions and transformation plans lazily.

Actions require Spark to execute the required data-processing work.
""")

In [0]:
# ------------------------------------------------------------
# Step 10 — Final Learning Summary
# ------------------------------------------------------------

print("=== FINAL LEARNING SUMMARY ===")

print("""
col():

F.col("column_name")

- Returns a Spark Column expression.
- Used to reference a column inside expressions.
- Useful with arithmetic, functions, cast(), alias(), etc.


alias():

F.col("old_name").alias("new_name")

- Gives the resulting Column expression a name.
- Can rename an existing column.
- Can also name a calculated expression.


cast():

F.col("column").cast("target_type")

- Converts a Column expression to another Spark data type.
- Commonly used to standardize Bronze source types
  into target/Silver types.
- Does not automatically guarantee data quality.


try_cast():

try_cast(value AS target_type)

- Attempts a conversion safely.
- Malformed values can become NULL instead of causing
  conversion failure.
- Resulting NULLs still need data-quality validation.


lit():

F.lit(value)

- Creates a Spark Column containing a constant value.
- Every output row receives that value.
- Useful for source, batch, audit, flag, and metadata fields.


Combined pattern:

col()
   ↓
Reference source column

cast()
   ↓
Standardize data type

alias()
   ↓
Standardize output name

lit()
   ↓
Add constant metadata

   ↓

Silver-ready column expressions
""")

print("=== S3.9 CELL 2 COMPLETED ===")

# S3.9 — Key Takeaways: col, alias, cast, lit

## 1. Four Core Building Blocks

| Function | Purpose | Example |
|---|---|---|
| `col()` | Reference a DataFrame column as a Spark Column expression | `F.col("order_id")` |
| `alias()` | Give a column or expression an output name | `F.col("o_orderkey").alias("order_id")` |
| `cast()` | Convert a column/expression to another data type | `F.col("price").cast("decimal(18,2)")` |
| `lit()` | Create a constant-value Spark column | `F.lit("TPCH").alias("source_system")` |

---

# 2. col()

`col()` references a DataFrame column as a Spark `Column` expression.

Example:

```python
F.col("o_totalprice")
```

This does not immediately retrieve a value.

It creates a column expression that can be used with:

```text
cast()
alias()
when()
round()
year()
arithmetic expressions
filter conditions
```

Example:

```python
F.col("o_totalprice") * 1.18
```

---

# 3. Ways to Reference Columns

### F.col()

```python
F.col("o_orderkey")
```

Useful when building reusable Spark expressions.

### DataFrame column syntax

```python
df["o_orderkey"]
```

Also returns a Spark `Column` expression.

### String column name

```python
"o_orderkey"
```

Many DataFrame APIs accept column-name strings.

Examples include:

```text
select()
drop()
groupBy()
orderBy()
```

### Important

Do not think of these as:

```text
correct vs incorrect
```

They are different ways of expressing column references depending
on the API and transformation being written.

---

# 4. alias()

`alias()` gives a Spark column or expression an output name.

### Rename an existing column

```python
F.col(
    "o_orderkey"
).alias(
    "order_id"
)
```

### Name a calculated expression

```python
(
    F.col("o_totalprice") * 1.18
).alias(
    "price_with_tax"
)
```

Therefore, `alias()` is not limited to renaming existing columns.

It can name any resulting Column expression.

---

# 5. cast()

`cast()` converts a Spark column/expression to another data type.

Example:

```python
F.col(
    "order_id"
).cast(
    "long"
)
```

Common Bronze → Silver conversions may include:

```text
STRING → BIGINT
STRING → DATE
STRING → TIMESTAMP
STRING → BOOLEAN
STRING → DECIMAL
```

Example:

```python
F.col(
    "total_price"
).cast(
    "decimal(18,2)"
)
```

### Important

The correct target data type depends on the business meaning.

For example:

```text
Identifiers
→ often BIGINT or STRING

Financial values
→ often DECIMAL when exact precision matters

Dates
→ DATE

Event timestamps
→ TIMESTAMP

Flags
→ BOOLEAN
```

Do not automatically convert every numeric value to `double`.

---

# 6. cast() Does Not Guarantee Data Quality

Suppose:

```text
order_id = "BAD_ID"
```

but we attempt:

```python
F.col(
    "order_id"
).cast(
    "long"
)
```

The source value is not valid for the requested type.

Behaviour can depend on the conversion and Spark/ANSI settings.

Therefore, do not assume:

```text
bad cast always → NULL
```

Some invalid conversions can raise errors.

---

# 7. try_cast() — Safer Conversion

`try_cast()` is useful when malformed values should become `NULL`
instead of failing the conversion.

Example:

```python
F.expr(
    "try_cast(order_id AS BIGINT)"
)
```

Conceptually:

```text
"1001"
   ↓
try_cast BIGINT
   ↓
1001
```

while:

```text
"BAD_ID"
   ↓
try_cast BIGINT
   ↓
NULL
```

### Important

`NULL` after `try_cast()` does not automatically mean the record
should be accepted.

It should normally lead to data-quality validation.

Example:

```text
try_cast()
    ↓
NULL produced
    ↓
Validate
    ↓
Accept / Reject / Quarantine
```

---

# 8. lit()

`lit()` creates a Spark Column containing a constant value.

Example:

```python
F.lit(
    "TPCH"
).alias(
    "source_system"
)
```

Every resulting row receives:

```text
source_system = TPCH
```

Typical uses include:

```text
source-system metadata
batch identifiers
pipeline versions
static flags
load metadata
environment identifiers
```

---

# 9. Metadata Example

Possible metadata columns could be:

```python
F.lit(
    "TPCH"
).alias(
    "source_system"
)
```

```python
F.lit(
    20261012
).alias(
    "batch_id"
)
```

```python
F.lit(
    False
).alias(
    "is_deleted"
)
```

For a date literal:

```python
F.lit(
    "2026-10-12"
).cast(
    "date"
).alias(
    "batch_date"
)
```

### Important

These are example audit/metadata fields.

Not every Silver table must contain exactly these columns.

The metadata design should follow the platform architecture,
lineage requirements, ingestion framework, and business requirements.

---

# 10. Combined Silver-Layer Pattern

A common expression pattern is:

```python
F.col(
    "o_orderkey"
).cast(
    "long"
).alias(
    "order_id"
)
```

Read it from left to right:

```text
col()
   ↓
Reference source column

cast()
   ↓
Standardize data type

alias()
   ↓
Standardize target name
```

Then metadata can be added with:

```python
F.lit(
    "TPCH"
).alias(
    "source_system"
)
```

---

# 11. Combined Example

```python
df_silver = df_orders.select(

    F.col(
        "o_orderkey"
    )
    .cast(
        "long"
    )
    .alias(
        "order_id"
    ),

    F.col(
        "o_custkey"
    )
    .cast(
        "long"
    )
    .alias(
        "customer_id"
    ),

    F.col(
        "o_totalprice"
    )
    .cast(
        "decimal(18,2)"
    )
    .alias(
        "total_price"
    ),

    F.col(
        "o_orderdate"
    )
    .cast(
        "date"
    )
    .alias(
        "order_date"
    ),

    F.lit(
        "TPCH"
    ).alias(
        "source_system"
    )
)
```

---

# 12. Bronze → Silver Interpretation

```text
Bronze source value
       ↓
col()
Reference source column
       ↓
cast()
Standardize data type
       ↓
alias()
Standardize target name
       ↓
lit()
Add required metadata
       ↓
Validate
       ↓
Silver-ready structure
```

---

# 13. Critical Rules

1. `col()` creates a Spark Column expression.

2. `alias()` can name an existing column or a calculated expression.

3. `cast()` changes the data type of a Column expression.

4. `cast()` does not guarantee that malformed source values are valid.

5. `try_cast()` is useful when failed supported conversions should return `NULL`.

6. `NULL` after conversion still requires data-quality validation.

7. `lit()` creates a constant-value column.

8. Metadata columns should be designed according to the pipeline requirements, not added blindly.

9. Use `DecimalType` for financial values when exact decimal precision is required.

10. These functions are building blocks used inside transformations such as `select()` and `withColumn()`.

---

# 14. Easy Way to Remember

```text
col()
→ Which source column?

cast()
→ What data type should it be?

alias()
→ What should the output column be called?

lit()
→ What constant value should every row receive?
```

---

# 15. Final S3.9 Understanding

```text
Source column
     ↓
col()
     ↓
cast()
     ↓
alias()
     ↓
Clean typed output

+

lit()
     ↓
Required constant metadata
```

**S3.9 — Core Column Expression Building Blocks completed.**